# Detecção de traços de reamostragem

Implementação do artigo:

[1] B. Mahdian e S. Saic, “Blind Authentication Using Periodic Properties of Interpolation”, IEEE Transactions on Information Forensics and Security, vol. 3, nº 3, p. 529–538, set. 2008, doi: 10.1109/TIFS.2004.924603.

Autor: Paulo Max Gil Innocencio Reis 

Email: paulo.pmgir@pf.gov.br

Serviço de Perícias em Audiovisual e Eletrônicos - INC/DITEC

*Reamostragem deixa traços periódicos na correlação entre pixels adjacentes*

# Importação de pacotes

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, transform
from scipy.fft import fft
from scipy.signal import correlate
from skimage.transform import radon
from ipywidgets import interact, interactive, fixed
from ipywidgets import widgets


# Primeria Parte: Criação dos widgets interativos
# Segunda Parte: Processamento para função de detecção de reamosatragem

#### *Periodicidades na autocovârianca da integração Radon das componentes em alta frequência*
#### *Detecção por Fourier*

In [11]:
def reamostragem(imagem):
    print("Função reamostragem iniciada")
    ordem = 2
    radon_passo = 1
    radon_max = 1
    im = None
    def selecionar_roi(Esquerda, Superior, Direita, Inferior):
        nonlocal im
        print(f"ROI selecionada: ({Esquerda}, {Superior}) to ({Direita}, {Inferior})")
        x1, x2 = min(Esquerda, Direita), max(Esquerda, Direita)
        y1, y2 = min(Superior, Inferior), max(Superior, Inferior)
        
        if imagem.ndim == 3:
            print("Imagem Colorida. Considerando somente canal de cor verde.")
            im = imagem[y1:y2, x1:x2, 1].astype(float)
        else:
            im = imagem[y1:y2, x1:x2].astype(float)
        
        print("Chamando processar_imagem()")
        processar_imagem()

    def processar_imagem():
        print("Função processar_imagem iniciada")
        if im is None:
            print("Por favor, selecione uma região de interesse primeiro.")
            return
        fig, axs = plt.subplots(3, 2, figsize=(15, 20))

        axs[0, 0].imshow(im, cmap='gray')
        axs[0, 0].axis('image')
        axs[0, 0].set_title('Imagem Original')

        theta = np.arange(0, radon_max + radon_passo, radon_passo)

        # Processamento na direção vertical
        d = np.diff(im, n=ordem, axis=0)
        axs[1, 0].imshow(d, cmap='gray')
        axs[1, 0].axis('image')
        axs[1, 0].set_title('Derivada Vertical')

        r = radon(np.abs(d), theta=theta, circle=False)
        r = np.diff(r, axis=0)

        ft = np.abs(fft(correlate(r[:, 0], r[:, 0], mode='full')))
        for i in range(r.shape[1]):
            ft = np.maximum(ft, np.abs(fft(correlate(r[:, i], r[:, i], mode='full'))))

        ft = ft / np.linalg.norm(ft)
        ll = len(ft)
        axs[2, 0].plot(np.arange(1/ll, 1 + 1/ll, 1/ll), ft)
        axs[2, 0].set_title('FFT da Covariância (Vertical)')
        for x in np.arange(0, 1.125, 0.125):
            axs[2, 0].axvline(x, color='r', linestyle='--')

        # Processamento na direção horizontal
        d = np.diff(im, n=ordem, axis=1)
        axs[1, 1].imshow(d, cmap='gray')
        axs[1, 1].axis('image')
        axs[1, 1].set_title('Derivada Horizontal')

        r = radon(np.abs(d), theta=theta, circle=False)
        r = np.diff(r, axis=0)

        ft = np.abs(fft(correlate(r[:, 0], r[:, 0], mode='full')))
        for i in range(r.shape[1]):
            ft = np.maximum(ft, np.abs(fft(correlate(r[:, i], r[:, i], mode='full'))))

        ft = ft / np.linalg.norm(ft)
        ll = len(ft)
        axs[2, 1].plot(np.arange(1/ll, 1 + 1/ll, 1/ll), ft)
        axs[2, 1].set_title('FFT da Covariância (Horizontal)')
        for x in np.arange(0, 1.125, 0.125):
            axs[2, 1].axvline(x, color='r', linestyle='--')

        plt.tight_layout()
        plt.show()

    print("Criando widget de interação. Selecione as margens")
    interact(selecionar_roi, 
             Esquerda=widgets.IntSlider(min=0, max=imagem.shape[1]-1, step=1, value=0),
             Superior=widgets.IntSlider(min=0, max=imagem.shape[0]-1, step=1, value=0),
             Direita=widgets.IntSlider(min=0, max=imagem.shape[1]-1, step=1, value=imagem.shape[1]-1),
             Inferior=widgets.IntSlider(min=0, max=imagem.shape[0]-1, step=1, value=imagem.shape[0]-1))

    print("Função reamostragem concluída")
    

# Entrada da imagem e chamada da função

In [12]:
imagem = io.imread('DLT_2961.jpg')

reamostragem(imagem)

Função reamostragem iniciada
Criando widget de interação. Selecione as margens


interactive(children=(IntSlider(value=0, description='Esquerda', max=2361), IntSlider(value=0, description='Su…

Função reamostragem concluída
